# S6E9: Rule-Mining at LB Resolution — a 10-Arm Ablation

The Playground S5E9/S6E9-style tabular competitions have a peculiar endgame: the
public leaderboard moves in steps of **0.00001**, the top-10 are separated by
**< 3e-4**, and the obvious "cheap" lever everyone reaches for is post-hoc
**business rules** mined from the train set (if income > X then nudge down, if
commute in [a,b] then nudge up, ...).

This notebook is a controlled experiment on whether that lever still works when
your base recipe is already near saturation. We took **one frozen base recipe**,
stacked **10 different single-rule arms** on it — one at a time, nothing else
changed — and scored each arm individually on the public LB.

**TL;DR — every arm tied or hurt the LB.**

| arm | rule sketch | train n | LB | Δ (×1e-5) |
|---|---|---|---|---|
| base | frozen recipe (no new rule) | — | 0.94651 | 0 |
| g1 | RangeAnxiety high band, −0.05 | 1,029 | 0.94650 | −1 |
| d1 | existing rule layer, graded ×0.0005 | 3,731 | 0.94651 | 0 |
| a1 | Subsidy=No & income<100k, −0.05 | 80,579 | 0.94647 | −4 |
| b2 | commute 65–83 km, −0.05 | 6,547 | 0.94636 | −15 |
| e1 | Subsidy=Yes & income<50k, −0.05 | 19,768 | 0.94624 | −27 |
| a2 | Subsidy=No & income<100k, **−1.0** | 80,579 | 0.94534 | −117 |
| c1 | income 156–170.5k, **+1.0** | 1,220 | 0.94398 | −253 |
| c2 | income 140–170.5k, **+0.30** | 7,866 | 0.94131 | −520 |
| b1 | commute 65–83 km, **−1.0** | 6,547 | 0.93783 | −868 |
| f1 | combo (c1 + b1) | — | 0.93557 | −1,094 |

Four lessons come out of this — the interesting part is that they are **not**
the lessons you'd predict from train-side statistics alone:

1. **Rule mining from train double-dips.** The base model already encodes the
   strong train signals; re-expressing them as rules is redundant at best.
2. **Hard blocks are catastrophic.** A ±1.0 shift in rank-01 space dominates the
   entire prediction scale; graded nudges of −0.05 are nearly invisible.
3. **Train signal depth ≠ LB damage.** The single deepest train anomaly
   (Subsidy=No & income<100k, 0.6% positive vs 17.5% global) was one of the
   *least* damaging arms — because the base model already handles it.
4. **The family saturates.** Once arms are all ≤ base, more rules from the same
   mine cannot help. The remaining gap to the top needs a *new signal source*,
   not a sharper chisel on the old one.

Everything below runs on the competition `train.csv` plus our open companion
dataset [`s6e9-realmlp-oof`](https://www.kaggle.com/datasets/defiaudit/s6e9-realmlp-oof)
(frozen base predictions + all 10 arm outputs).

In [ ]:
import numpy as np, pandas as pd, glob
import matplotlib.pyplot as plt

def find(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    assert hits, f'{name} not mounted'
    return hits[0]

BG, CARD, INK, MUT = '#0b1020', '#131a2e', '#e8edf7', '#9aa7c7'
BLUE, RED, GREEN, AMBER = '#8ab4f8', '#ff5c5c', '#3fb950', '#f9ab00'
plt.rcParams.update({'figure.facecolor': BG, 'axes.facecolor': BG,
                     'axes.edgecolor': '#2a3557', 'axes.labelcolor': INK,
                     'xtick.color': MUT, 'ytick.color': MUT, 'text.color': INK,
                     'grid.color': '#1d2742', 'font.size': 10.5})

tr = pd.read_csv(find('train.csv'))
print('train:', tr.shape, '| global positive rate:', round(tr['Will_Buy_EV'].mean(), 4))
tr.head(3)

## 1. The resolution arithmetic

Why do rules behave so differently here than in a normal competition? Because of
what a leaderboard metric at 1e-5 resolution actually sees. Our base predictions
are rank-normalized to [0, 1] over ~125,690 test rows, so the **median gap
between two adjacent sorted predictions** is tiny:

In [ ]:
base = pd.read_csv(find('r6_rebuilt_94651.csv'))
p = np.sort(base['Will_Buy_EV'].values)
gaps = np.diff(p)
print(f'rows: {len(p):,}')
print(f'median adjacent gap: {np.median(gaps):.2e}')
print(f'90th-pct adjacent gap: {np.quantile(gaps, .9):.2e}')
print(f'max adjacent gap: {gaps.max():.2e}')

A graded rule of **−0.05** applied to n rows moves each of those rows down by
0.05 × 125,690 ≈ **6,285 rank positions** — but that is still a *small, smooth*
perturbation the metric partially absorbs. A hard block of **−1.0** moves rows by
125,690 positions: the entire test set, i.e. those rows jump to the very bottom
of the ranking. If the block catches even a modest share of positives, the
ranking loss is massive. That is the mechanism behind the a2/b1/f1 columns in the
table above — and the dose–response inside each arm pair (a1 vs a2, b2 vs b1) is
purely the shift size, nothing else changes.

This is the first practical takeaway: **in rank-space ensembles, rule magnitudes
are not linear — they are categorical.** −0.05 and −1.0 are not the same rule
with different confidence; they are different *species*.

## 2. Mining methodology — a vectorized band scan

The scan we used is deliberately boring and fast: bin every candidate column
(quantile bins for numerics, categories as-is), group, and compute

- `n` — band size,
- `rate` — positive rate inside the band,
- `dev` — `rate − global_rate`, the raw deviation,
- and a minimum-size guard (`n ≥ 300`) so we don't chase noise.

No per-bin Python loops, no significance theatrics — on 120k rows a single
`groupby` per column is enough to see everything the data has to say.

In [ ]:
GLOBAL = tr['Will_Buy_EV'].mean()

def band_scan(col, bins=None, q=20, min_n=300):
    if bins is not None:
        b = pd.cut(tr[col], bins)
    elif tr[col].dtype == object or tr[col].nunique() <= 10:
        b = tr[col]
    else:
        b = pd.qcut(tr[col], q, duplicates='drop')
    g = tr.groupby(b, observed=True)['Will_Buy_EV'].agg(['size', 'mean'])
    g = g[g['size'] >= min_n].copy()
    g['dev'] = g['mean'] - GLOBAL
    g['lift'] = g['mean'] / GLOBAL
    return g.sort_values('dev')

num_cols = ['Annual_Income_USD', 'Daily_Commute_km', 'Age']
cat_cols = ['Subsidy_Available', 'Range_Anxiety_Level',
            'Environmental_Concern_Level', 'City_Type', 'Current_Car_Type']

tables = {}
for c in num_cols:
    tables[c] = band_scan(c, q=24)
for c in cat_cols:
    tables[c] = band_scan(c)

for c, t in tables.items():
    print(f'--- {c} (worst 3 / best 3 bands) ---')
    sel = pd.concat([t.head(3), t.tail(3)])
    print(sel.to_string(float_format=lambda v: f'{v:,.4f}'))
    print()

## 3. What the scan says — and the trap it sets

The scan surfaces three genuine anomalies and one instructive null:

- **Subsidy=No & income<100k** — the monster. 0.6% positive vs 17.5% global
  (a ~29× suppression), n≈80,579. Any miner will pounce on this.
- **Commute 65–83 km valley** — dev −0.12…−0.14 across several bands.
- **Income 140–170.5k shelf** — visibly elevated positive rate.
- **Age** — nothing. Clean null across all bins; we never spent an arm on it.

The trap: these are exactly the anomalies the base *model* has already fit. A
gradient-boosted or linear model trained on this table **must** have absorbed a
29× subgroup suppression — it's the strongest single interaction in the data.
Stacking a rule that re-implements it does not add information; it adds
**stale, step-shaped noise** on top of a smooth, already-optimal fit. That is
lesson 1 in action, and the a1 result (−4e-5 despite n=80,579) is its receipt.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4), constrained_layout=True)

t = tables['Daily_Commute_km'].reset_index()
x = np.arange(len(t))
colors = [RED if d < 0 else BLUE for d in t['dev']]
axes[0].bar(x, t['dev'] * 100, color=colors, width=.75)
axes[0].axhline(0, color=MUT, lw=.8)
axes[0].set_title('Deviation vs global rate by commute band (pp)', loc='left', color=INK)
axes[0].set_xlabel('Daily_Commute_km quantile band'); axes[0].set_ylabel('dev (pp)')
axes[0].grid(axis='y', alpha=.5)

t2 = tables['Annual_Income_USD'].reset_index()
x2 = np.arange(len(t2))
colors2 = [GREEN if d > 0 else BLUE for d in t2['dev']]
axes[1].bar(x2, t2['dev'] * 100, color=colors2, width=.75)
axes[1].axhline(0, color=MUT, lw=.8)
axes[1].set_title('Deviation vs global rate by income band (pp)', loc='left', color=INK)
axes[1].set_xlabel('Annual_Income_USD quantile band'); axes[1].set_ylabel('dev (pp)')
axes[1].grid(axis='y', alpha=.5)
plt.show()

sub = tr.groupby('Subsidy_Available')['Will_Buy_EV'].agg(['size', 'mean'])
print(sub.to_string(float_format=lambda v: f'{v:,.4f}'))

## 4. The 10 arms — design

Each arm is **one** rule applied to the frozen base predictions, everything else
identical. Magnitudes come in two species (graded −0.05 / +0.30, and hard −1.0 /
+1.0) precisely to measure the dose–response described in §1. Arms:

| arm | rule sketch (applied to test rows) | species |
|---|---|---|
| a1 | Subsidy=No & income<100k → −0.05 | graded |
| a2 | same band → −1.0 | hard |
| b1 | commute ∈ [65, 83] → −1.0 | hard |
| b2 | commute ∈ [65, 83] → −0.05 | graded |
| c1 | income ∈ [156k, 170.5k] → +1.0 | hard (up) |
| c2 | income ∈ [140k, 170.5k] → +0.30 | graded (up) |
| d1 | existing 4-rule layer, each block scaled ×0.0005 | graded |
| e1 | Subsidy=Yes & income<50k → −0.05 | graded |
| f1 | c1 + b1 stacked | hard combo |
| g1 | RangeAnxiety high band → −0.05 | graded |

The results table in the TL;DR is the full outcome — sorted by damage. Two
patterns deserve a pause:

- **Within-pair dose–response is brutal and consistent**: a1→a2 is 29× worse
  with the same band, b2→b1 is 58× worse. Species matters, not size.
- **Across-band ordering is inverted vs train depth**: the 80k-row subsidy arm
  (deepest train signal) lost 4e-5; the 6.5k-row commute arm lost 868e-5. The
  variable that correlates with damage is *how well the base model already fits
  the band*, not how loud the band is in train.

In [ ]:
arms = [
    ('f1 combo hard',        -1094, RED),
    ('b1 commute hard',       -868, RED),
    ('c2 income graded .3',   -520, RED),
    ('c1 income hard up',     -253, RED),
    ('a2 subsidy hard',       -117, RED),
    ('e1 subyes graded',       -27, AMBER),
    ('b2 commute graded',      -15, AMBER),
    ('a1 subsidy graded',       -4, AMBER),
    ('g1 anxiety graded',       -1, AMBER),
    ('d1 rules rescaled',       0, GREEN),
]
arms = arms[::-1]
fig, ax = plt.subplots(figsize=(11, 5.2), constrained_layout=True)
y = np.arange(len(arms))
vals = [a[1] for a in arms]
ax.barh(y, vals, color=[a[2] for a in arms], height=.62)
ax.set_yticks(y); ax.set_yticklabels([a[0] for a in arms])
ax.axvline(0, color=MUT, lw=1)
for i, v in enumerate(vals):
    ax.text(v - 18, i, f'{v:+,}', va='center', ha='right' if v < 0 else 'left',
            color=MUT, fontsize=9)
ax.set_title('LB delta vs frozen base (×1e-5) — every arm ≤ 0', loc='left', color=INK)
ax.set_xlabel('Δ LB (×1e-5)'); ax.grid(axis='x', alpha=.5)
plt.show()

## 5. The harness — how an arm is applied (runnable)

This is the exact transformation each arm ran through: merge test features into
the frozen predictions, apply the rule, re-rank to [0, 1]. The cell below builds
arm **b2** (commute soft) end-to-end; swapping the rule dict gives any other arm
from the paper table. All 10 outputs ship as flat CSVs in the companion dataset (`r7_*.csv`).

In [ ]:
base = pd.read_csv(find('r6_rebuilt_94651.csv'))
test = pd.read_csv(find('test.csv'))
df = base.merge(test[['id', 'Daily_Commute_km']], on='id', how='left')

RULE = {'col': 'Daily_Commute_km', 'lo': 65.0, 'hi': 83.0, 'delta': -0.05}
mask = (df[RULE['col']] >= RULE['lo']) & (df[RULE['col']] <= RULE['hi'])
print('rows hit by rule:', int(mask.sum()))

out = df[['id', 'Will_Buy_EV']].copy()
out.loc[mask, 'Will_Buy_EV'] = out.loc[mask, 'Will_Buy_EV'] + RULE['delta']

# rank-01 normalize (monotone, order-preserving)
out['Will_Buy_EV'] = (out['Will_Buy_EV'].rank(method='average')
                      / len(out)).clip(0, 1)
out.to_csv('arm_b2_demo.csv', index=False)
print(out.head(3))
print('saved arm_b2_demo.csv — scored 0.94636 on the public LB (−15e-5 vs base)')

## 6. Lessons — generalized

1. **Double-dipping.** If a signal is strong enough to find by band-mining, a
   trained model has already encoded it (better, smoother, with interactions).
   Post-hoc rules are only additive where the model is *structurally blind* —
   e.g. genuinely new external features, not re-binnings of train columns.
2. **Species, not magnitude.** In rank-space, keep every hand-crafted delta
   within the graded regime (|Δ| ≤ ~0.05) or accept that you are re-sorting the
   entire leaderboard row set. Hard blocks are a footgun unless you can prove
   the band contains *zero* positives.
3. **Evaluate the nulls too.** Age's clean null saved us an arm. Scan everything,
   but only spend LB submissions where the anomaly is both large *and* plausibly
   under-fitted by the base model.
4. **Saturation discipline.** When 10 consecutive arms land ≤ base, the family
   is done. The correct next move is a different signal source (new features,
   new model family, external data at low blend weight) — not a sharper chisel.

## Reproducibility

- Frozen base recipe: two-model rank blend, reproduced from a public 0.946-line
  recipe; predictions ship as `r6_rebuilt_94651.csv` in the companion dataset.
- All 10 arm outputs (the exact files submitted) ship as `r7_out.zip`.
- LB deltas are from the public leaderboard, September 2026; the metric and
  split are fixed for the competition, so the table stands as a permanent result.

If this helped you calibrate your own endgame rules, a vote helps others find it.
Critique welcome in the comments — especially if you've seen a *counter*-example
where a graded rule beat its base on a saturated playground.